# 실습 1. 청킹 전략 비교 — `chunk_size` / `overlap`

## 실습 목표

RAG에서 **검색의 단위는 '청크'** 입니다(교안 1.2). 같은 문서라도 어떻게 자르느냐에 따라
청크 수와 검색 결과가 달라집니다. 이 실습에서는 `chunk_size`/`overlap` 을 바꿔가며
**"청크가 크면 정밀도↓, 작으면 문맥↓"** 의 트레이드오프를 눈으로 확인합니다.

- 문서를 여러 `chunk_size` 로 분할해 **청크 수·평균 길이** 비교
- 같은 질문으로 검색해 **무엇이 검색되는지** 관찰
- 정답 청크 크기는 데이터마다 다르다 → **실험으로 정한다**

## 1. 준비 — 문서·임베딩 로드

`_common.py` 의 헬퍼를 재사용합니다. 한국어 샘플 문서 10개와 로컬 HuggingFace 임베딩
(`multilingual-e5-small`, 한국어 양호)을 불러옵니다.

In [ ]:
from _common import sample_documents, get_embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

docs = sample_documents()
embeddings = get_embeddings()
QUESTION = "임베딩 모델이 한국어 검색 품질에 주는 영향은?"
print(f"원본 문서 {len(docs)}개 로드 완료 · 질문: {QUESTION}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7570.88it/s]


원본 문서 10개 로드 완료 · 질문: 임베딩 모델이 한국어 검색 품질에 주는 영향은?


## 2. `chunk_size` 를 바꿔가며 청크 수·검색 비교 (교안 1.2)

`RecursiveCharacterTextSplitter` 는 `\n\n`(문단)→`\n`(줄)→`' '`(단어) 순으로 **자연 경계부터**
분할을 시도합니다. `overlap` 은 인접 청크를 겹쳐 경계에서 잘린 문맥을 보완합니다(보통 10~20%).

In [6]:
for size, overlap in [(120, 0), (300, 50), (800, 100)]:
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap).split_documents(docs)
    vs = Chroma.from_documents(chunks, embeddings, collection_name=f"day21_chunk_{size}")
    hits = vs.as_retriever(search_kwargs={"k": 2}).invoke(QUESTION)

    avg_len = sum(len(c.page_content) for c in chunks) / max(len(chunks), 1)
    print(f"■ chunk_size={size:>3}, overlap={overlap:>3} "
          f"→ 청크 {len(chunks):>2}개, 평균 길이 {avg_len:.0f}자")
    for d in hits:
        print(f"     · ({d.metadata['topic']}) {d.page_content[:50]}…")
    vs.delete_collection()
    print()

■ chunk_size=120, overlap=  0 → 청크 20개, 평균 길이 75자
     · (임베딩) 특화 임베딩 모델을 써야 검색 정확도가 올라간다.…
     · (임베딩) 임베딩은 텍스트를 의미가 보존된 고정 길이 벡터로 변환한 것이다. 비슷한 의미의 문장은 벡…

■ chunk_size=300, overlap= 50 → 청크 10개, 평균 길이 152자
     · (임베딩) 임베딩은 텍스트를 의미가 보존된 고정 길이 벡터로 변환한 것이다. 비슷한 의미의 문장은 벡…
     · (LLM) 대규모 언어 모델(LLM)은 방대한 텍스트로 사전학습된 트랜스포머 기반 모델로, 다음 토큰…

■ chunk_size=800, overlap=100 → 청크 10개, 평균 길이 152자
     · (임베딩) 임베딩은 텍스트를 의미가 보존된 고정 길이 벡터로 변환한 것이다. 비슷한 의미의 문장은 벡…
     · (LLM) 대규모 언어 모델(LLM)은 방대한 텍스트로 사전학습된 트랜스포머 기반 모델로, 다음 토큰…



**포인트**

- 작은 청크(120): 청크 수↑, 한 청크 문맥이 짧아 **답에 필요한 정보가 잘릴** 수 있음
- 큰 청크(800): 청크 수↓, 한 청크에 여러 주제가 섞여 **검색 정밀도↓**
- 본 데모는 문서가 짧아 300·800이 비슷하지만, **긴 실데이터일수록 큰 청크의 주제 혼합**이 두드러짐
- "정답 크기"는 데이터마다 다르다 → **반드시 실험으로 정한다**(이 셀이 그 실험 틀)

# [TODO]

지금까지 본 패턴을 직접 응용해 봅니다.

## [TODO] 1. 새로운 `(chunk_size, overlap)` 조합 추가 실험

위 비교 리스트에 **`(500, 80)`** 조합을 추가해, 청크 수·평균 길이가 300/800 사이 어디에
위치하는지 확인하세요. (리스트에 튜플 하나만 추가하면 됩니다.)

In [7]:
combos = [(120, 0), (300, 50), (800, 100)]
# [TODO] 1: (500, 80) 조합을 combos 에 추가
# [TODO] 여기에 코드를 작성하세요.
combos.append((500, 80))

for size, overlap in combos:
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap).split_documents(docs)
    avg_len = sum(len(c.page_content) for c in chunks) / max(len(chunks), 1)
    print(f"chunk_size={size:>3}, overlap={overlap:>3} → 청크 {len(chunks):>2}개, 평균 {avg_len:.0f}자")

chunk_size=120, overlap=  0 → 청크 20개, 평균 75자
chunk_size=300, overlap= 50 → 청크 10개, 평균 152자
chunk_size=800, overlap=100 → 청크 10개, 평균 152자
chunk_size=500, overlap= 80 → 청크 10개, 평균 152자


## [TODO] 2. 특정 `chunk_size` 의 top-1 검색 주제를 반환하는 함수

`top1_topic(size, overlap, question)` 함수를 작성하세요.
- 주어진 `size`/`overlap` 으로 문서를 분할·인덱싱
- `question` 으로 검색해 **top-1 청크의 `metadata['topic']`** 을 반환
- (힌트) 위 2번 셀의 인덱싱·검색 코드를 함수로 감싸면 됩니다

In [9]:
def top1_topic(size, overlap, question):
    # [TODO] 2: size/overlap 으로 분할·인덱싱 후 top-1 청크의 topic 반환
    # [TODO] 여기에 코드를 작성하세요.
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap).split_documents(docs)
    vs = Chroma.from_documents(chunks, embeddings, collection_name=f"day21_chunk_{size}")
    hits = vs.as_retriever(search_kwargs={"k": 1}).invoke(question)
    top1_topic = hits[0].metadata['topic'] # if hits else None
    return top1_topic

print("top-1 주제:", top1_topic(300, 50, QUESTION))

top-1 주제: 임베딩


## 실습 정리

- `chunk_size`/`overlap` 이 **청크 수·평균 길이·검색 결과** 를 어떻게 바꾸는지 직접 확인
- 작은 청크(문맥↓) vs 큰 청크(정밀도↓)의 트레이드오프 체감
- **청크 크기는 데이터로 튜닝** — 이 노트북이 그 실험 틀